# Long-Term Environmental Measurements from Estuaries and Coastal Zones of the Basque Country, 1995–2014 Exploration with `mlcroissant`
This notebook provides a template and practical example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.
- Croissant schema: [https://sen.science/doi/10.71728/senscience.kv2w-204a/fair2.json](https://sen.science/doi/10.71728/senscience.kv2w-204a/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.kv2w-204a/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access structured metadata
metadata = dataset.metadata  # metadata is a single Dataset object

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Temporal Coverage:", metadata.temporalCoverage)
print("Spatial Coverage:", metadata.spatialCoverage)
print("Version:", metadata.version)
print("Authors (@id):", [author['@id'] for author in metadata.author])
print("Record Set IDs:", [rs['@id'] for rs in getattr(metadata, 'recordSet', [])])


## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the available record sets along with their `@id`. For each record set, we also print details of fields and columns using their `@id`.

In [ ]:
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets or len(record_sets) == 0:
    print("No record sets found in metadata.")
else:
    # Display each record set with its @id and fields
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Record Set @id: {rs_id}")
        # Attempt to fetch record set metadata from croissant
        try:
            rs_meta = dataset.record_set(rs_id)
            if hasattr(rs_meta, 'field'):
                print("  Fields:")
                for field in rs_meta.field:
                    print(f"    Field @id: {field['@id']} Name: {field['name']} DataType: {field.get('dataType','')}")
                    if 'column' in field:
                        print("      Columns:")
                        for col in field['column']:
                            print(f"        Column @id: {col['@id']}")
        except Exception as e:
            print("  Could not fetch record set details.", e)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below we select the first available record set as example (replace with your preferred `@id` if desired).

In [ ]:
dataframes = {}

# Collect record set IDs
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]

if record_set_ids:
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            # Create DataFrame with all fields (by @id)
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set {rs_id}: columns are {df.columns.tolist()[:10]}")
        except Exception as e:
            print(f"Error loading record set {rs_id}: {e}")
    # Example: show head of first record set
    first_rs_id = record_set_ids[0]
    print(f"\nHead of first record set ({first_rs_id}):")
    print(dataframes[first_rs_id].head())
else:
    print("No record sets found to extract data.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we select a numeric field by its `@id` and demonstrate filtering and normalization.
If columns are not available or not numeric, please adapt the code to your dataset specifics.

In [ ]:
# Choose first record set and inspect its fields
from pandas.api.types import is_numeric_dtype

if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    # Find numeric field by @id
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Use the first numeric field
        print(f"Using numeric field @id: {numeric_field}")
        threshold = df[numeric_field].median()  # set threshold automatically
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field if available
        group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field @id: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Grouped data by {group_field}: (showing means)")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets found for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following code demonstrates histogram and scatter plot visualizations for one numeric and one categorical field by their `@id`.

*You may need to adapt the field `@id`s to your dataset as shown above.*

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and numeric_fields:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    numeric_field = numeric_fields[0]
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Scatter plot with categorical field if available
    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(8,4))
        plt.scatter(df[group_field], df[numeric_field], alpha=0.5)
        plt.title(f"{numeric_field} vs {group_field} (@id)")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides rich environmental records suitable for ecosystem, trend, and spatial analyses.
- We demonstrated how to load and inspect metadata and record sets using their `@id` using `mlcroissant`.
- Numeric fields were explored and visualizations produced to facilitate further scientific analysis.
- For full research studies, further domain-specific filtering, imputation, and visualization steps will be required.

For more detail on variables, data sources, or methods, see documentation linked in the Croissant schema.